<a href="https://colab.research.google.com/github/alexklupsch/wur_deep_learning/blob/main/multi_label.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi Label Image Classification -- UCM airborne images -- Land Cover Classification

This notebook goes through our first attempt at handling the problem of a multi label classification of airborne images.

Inputs are 2100 images of the size 256x256p with a resolution of 0.3m, labeled by hand with 17 land cover classes. It is a relabeling of the original UCM dataset which consisted of a 100 images per single label class for 21 classes.

## 1. Data Loading and Preparation

In [ ]:
!pip install lightning

In [ ]:
import os
import zipfile
import random
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from torchvision.transforms import ToTensor
from PIL import Image
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torchvision.models as models
from lightning.pytorch import LightningModule, Trainer
from lightning.pytorch.callbacks import BackboneFinetuning
from lightning.pytorch.loggers import WandbLogger

In [ ]:
## loading the data and directory handling
! git clone https://git.wur.nl/lobry001/ucmdata.git
os.chdir('ucmdata')

with zipfile.ZipFile('UCMerced_LandUse.zip', 'r') as zip_ref:
  zip_ref.extractall('UCMImages')

!mv UCMImages/UCMerced_LandUse/Images .
!rm -rf UCMImages README.md UCMerced_LandUse.zip
!ls

UCM_images_path = "Images/"
Multilabels_path = "LandUse_Multilabeled.txt"

In [ ]:
import pandas as pd

# loading the labels
path = "/content/ucmdata/LandUse_Multilabeled.txt"
df = pd.read_csv(path, sep="\t", header=0)

In [ ]:
df.head()

In [ ]:
import seaborn as sns
# calc and plot the class count -> taken from medium post
class_count = pd.DataFrame(df.sum(axis=0)).reset_index()
class_count.columns = ["class", "Count"]
class_count.drop(class_count.index[0], inplace=True)
class_count = class_count.sort_values(by="Count", ascending=False)

sns.barplot(class_count, x="Count", y="class")


In [ ]:
## Accessing the images
dir = "/content/ucmdata/Images"
image_dict = {} # dict for key:class, value: [image_paths]
labels_map = {} # dict for key:class_num, value: class_name

for class_num, class_name in enumerate(sorted(os.listdir(dir))):
  labels_map[class_num] = class_name
  class_dir = os.path.join(dir, class_name)

  if os.path.isdir(class_dir): # sort out *.txt-file
    for image in sorted(os.listdir(class_dir)):
      image_path = os.path.join(class_dir, image)
      df.loc[df["IMAGE\\LABEL"] == image.split(sep=".")[0], ["path"]] = image_path # string matching, then save path in column

In [ ]:
print(df["path"].notna().all())
df.head()

In [ ]:
# reduce dataset to one hot encoding vectors
label_cols = df.columns.difference(["IMAGE\\LABEL", "path"])

# transfrom
labels = df[label_cols].values
paths = df["path"].values
dataset = [(path, label) for label, path in zip(labels, paths)]

In [ ]:
def split_train_test(dataset, val_ratio, test_ratio):

  random.seed(42) # to ensure the same test set for different models
  train_dataset = []
  val_dataset = []
  test_dataset = []

  random.shuffle(dataset)
  sample_length = len(dataset)

  test_split = int(sample_length * test_ratio)
  test_dataset.extend([sample for sample in dataset[:test_split]])

  dataset = dataset[test_split:]

  var_split = int(sample_length * val_ratio)
  val_dataset.extend([sample for sample in dataset[:var_split]])
  train_dataset.extend([sample for sample in dataset[var_split:]])

  return train_dataset, val_dataset, test_dataset

train_dataset,val_dataset, test_dataset = split_train_test(dataset, val_ratio = 0.15, test_ratio=0.15)

In [ ]:
def plot_random_samples(data, n_samples=20, cols=5, figsize=(15, 3)):
    """
    Plot random images from dataset (ignores labels).

    Parameters
    ----------
    data : list of tuples
        Each item is (image_path, label_vector)
    n_samples : int
        Number of random images to plot
    cols : int
        Number of subplot columns
    figsize : tuple
        Base figure size (width, height_per_row)
    """
    import random
    import matplotlib.pyplot as plt
    from PIL import Image

    # Randomly sample data points
    samples = random.sample(data, min(n_samples, len(data)))

    rows = (len(samples) + cols - 1) // cols
    plt.figure(figsize=(figsize[0], figsize[1] * rows))

    for i, (image_path, label_vector) in enumerate(samples):
        with Image.open(image_path) as img:
            img = img.convert("RGB")

            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)

            active_labels = [label_cols[j] for j, v in enumerate(label_vector) if v == 1]
            plt.title(", ".join(active_labels), fontsize=8)

            plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_random_samples(test_dataset) ## should plot the same images in every project!!

In [ ]:
class UCMDatasetMultiLabel(Dataset):
#Custom Dataset for our images
  def __init__(self, samples_list, transform=None):
    self.samples = samples_list
    self.transform = transform

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, index):
    img_path = self.samples[index][0]
    image = Image.open(img_path)
    label = self.samples[index][1]
    if self.transform:
      image = self.transform(image)
    return image, label

In [ ]:
## define the transforms
transforms_aug = v2.Compose([
    v2.Resize((224, 224)), ## valid resnet image size, fix passing size as tuple
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(degrees=(-10, 10)),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean= [0.485, 0.456,0.406],
                 std=[0.229, 0.224, 0.225])  ## imagenet mean and std
])

transforms = v2.Compose([
        v2.Resize((224, 224)), ## valid resnet image size, fix passing size as tuple
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean= [0.485, 0.456,0.406],
                     std=[0.229, 0.224, 0.225])  ## imagenet mean and std
])

train_ds_aug = UCMDatasetMultiLabel(train_dataset, transform=transforms_aug)
train_ds = UCMDatasetMultiLabel(train_dataset, transform=transforms)
val_ds = UCMDatasetMultiLabel(val_dataset, transform=transforms)
test_ds = UCMDatasetMultiLabel(test_dataset, transform=transforms)

train_dataloader_aug = DataLoader(train_ds_aug, batch_size=32, shuffle=True)
train_dataloader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_dataloader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
# Display image and label.
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")

img = train_features[0].squeeze()
label = train_labels[0]

### Unnormalize if wanted
# mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
# std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
# img = img * std + mean
# img = torch.clamp(img, 0, 1)

# Convert label vector to class names
active_indices = (label == 1).nonzero(as_tuple=True)[0].tolist()
label_names = [label_cols[i] for i in active_indices]

# Plot
plt.imshow(img.permute(1, 2, 0))
plt.title(", ".join(label_names), fontsize=10)
plt.axis("off")
plt.show()

print(f"Raw label vector: {label}")

In [ ]:
# setting up weights and biases
import wandb
wandb.login(relogin =True)

In [ ]:
from lightning.pytorch.callbacks import early_stopping
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

class ResNetClassifier(LightningModule):
    def __init__(self, num_classes=17, learning_rate=4e-3, threshold=0.5):
        super().__init__()
        self.save_hyperparameters()

        resnet = models.resnet50(weights="DEFAULT")
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(resnet.fc.in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)  # raw logits

    def _shared_step(self, batch, stage):
        x, y = batch
        y = y.float()  # important for BCEWithLogitsLoss
        logits = self(x)

        # Multi-label loss
        loss = nn.functional.binary_cross_entropy_with_logits(logits, y)

        # Convert logits to probabilities
        probs = torch.sigmoid(logits)

        # Convert probabilities to binary predictions
        preds = (probs >= self.hparams.threshold).float()

        # Simple element-wise accuracy
        acc = (preds == y).float().mean()

        self.log(f"{stage}_loss", loss, prog_bar=True)
        self.log(f"{stage}_acc", acc, prog_bar=True)

        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        return self._shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        probs = torch.sigmoid(logits)
        preds = (probs >= self.hparams.threshold).int()
        return preds, probs, y

    def configure_optimizers(self):
        return torch.optim.Adam(self.head.parameters(), lr=self.hparams.learning_rate)


# Setup the finetuning callback
backbone_finetuning = BackboneFinetuning(
    unfreeze_backbone_at_epoch=20,  # Start unfreezing backbone at epoch 10
    lambda_func=lambda epoch: 1.5,  # Gradually increase backbone learning rate
    backbone_initial_ratio_lr=0.01,  # Backbone starts at 10% of head learning rate
    should_align=True,  # Align rates when backbone rate reaches head rate
    verbose=True,  # Print learning rates during training
)

early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min")

In [ ]:
# Close any previous run from the notebook session
if wandb.run is not None:
    wandb.finish()

# Create a brand-new W&B run
wandb_logger = WandbLogger(
    project="multilabel",
    name=None,          # optional: let W&B auto-name it
    #log_model=True     # optional
)

wandb_logger.experiment.config.update({
    "Dataset": "UCM multi label",
    "Augmentation": "Yes - Fixed for only Train - Added CenterCrop",
    "Early Stopping": "Yes at 3",
    "Epochs": 15,
    "Model": "Resnet50"
})

model = ResNetClassifier()

trainer = Trainer(
    callbacks=[backbone_finetuning, early_stopping],
    max_epochs=15,
    logger=wandb_logger
)

trainer.fit(model, train_dataloader, val_dataloader)

# Mark this run finished so the next rerun starts a new one
#wandb.finish()

In [ ]:
# use seafile for confusion matrix
#trainer.test(dataloaders=test_dataloader)
preds = trainer.predict(model, val_dataloader)

all_preds = torch.cat([p[0] for p in preds])
all_targets = torch.cat([p[2] for p in preds])

# Convert logits → probabilities → binary predictions
y_pred = all_preds.int()
y_true = all_targets.int()

In [ ]:
preds[0]

In [ ]:
print(next(iter(zip(all_preds, all_targets.int()))))

In [ ]:
for i, class_name in enumerate(label_cols):
    y_t = y_true[:, i]
    y_p = y_pred[:, i]

    TP = ((y_p == 1) & (y_t == 1)).sum().item()
    TN = ((y_p == 0) & (y_t == 0)).sum().item()
    FP = ((y_p == 1) & (y_t == 0)).sum().item()
    FN = ((y_p == 0) & (y_t == 1)).sum().item()

    accuracy = (TP + TN) / (TP + TN + FP + FN + 1e-8)
    recall = TP / (TP + FN + 1e-8)   # how many positives you found
    precision = TP / (TP + FP + 1e-8)

    print(f"\nClass: {class_name}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

In [ ]:


# =========================
# Overall metrics
# =========================

# Element-wise accuracy
overall_accuracy = (y_pred == y_true).float().mean().item()

# Subset (exact match) accuracy
subset_accuracy = (y_pred == y_true).all(dim=1).float().mean().item()

print(f"Overall element-wise accuracy: {overall_accuracy:.4f}")
print(f"Subset accuracy (exact match): {subset_accuracy:.4f}")

# =========================
# Per-class accuracy
# =========================

per_class_accuracy = (y_pred == y_true).float().mean(dim=0)

print("\nPer-class accuracy:")
for i, class_name in enumerate(label_cols):
    print(f"{class_name}: {per_class_accuracy[i].item():.4f}")

# =========================
# Optional: better metrics (recommended)
# =========================

from sklearn.metrics import classification_report

print("\nClassification report:")
print(classification_report(
    y_true.numpy(),
    y_pred.numpy(),
    target_names=label_cols,
    zero_division=0
))

In [ ]:
def plot_mismatches(
    model,
    dataloader,
    label_cols,
    device,
    threshold=0.5,
    max_images=12,
    cols=3,
    figsize=(15, 5),
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
):
    """
    Plot misclassified multi-label samples with true and predicted labels.

    Parameters
    ----------
    model : torch.nn.Module
        Trained model.
    dataloader : torch.utils.data.DataLoader
        Dataloader yielding (images, labels).
    label_cols : list-like
        Class names in the same order as the multi-hot label vectors.
    device : torch.device or str
        Device used for inference.
    threshold : float
        Sigmoid threshold for turning probabilities into binary predictions.
    max_images : int
        Maximum number of mismatched images to display.
    cols : int
        Number of subplot columns.
    figsize : tuple
        Base figure size (width, height_per_row).
    mean, std : tuple
        Normalization stats used for unnormalizing images before plotting.
    """
    import math
    import torch
    import matplotlib.pyplot as plt

    model.eval()
    model.to(device)

    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)

    mismatches = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).int()

            # Find samples where full label vector does not match
            mismatch_mask = ~(preds == y.int()).all(dim=1)
            mismatch_indices = mismatch_mask.nonzero(as_tuple=True)[0]

            for idx in mismatch_indices:
                mismatches.append((
                    x[idx].cpu(),
                    y[idx].cpu().int(),
                    preds[idx].cpu().int()
                ))

                if len(mismatches) >= max_images:
                    break

            if len(mismatches) >= max_images:
                break

    if len(mismatches) == 0:
        print("No mismatches found.")
        return

    rows = math.ceil(len(mismatches) / cols)
    plt.figure(figsize=(figsize[0], figsize[1] * rows))

    for i, (img, y_true, y_pred) in enumerate(mismatches):
        # Unnormalize image
        img = img * std + mean
        img = torch.clamp(img, 0, 1)

        # Convert label vectors to names
        true_names = [label_cols[j] for j, v in enumerate(y_true) if v == 1]
        pred_names = [label_cols[j] for j, v in enumerate(y_pred) if v == 1]

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img.permute(1, 2, 0))
        plt.axis("off")
        plt.title(
            "True: " + (", ".join(true_names) if true_names else "none") +
            "\nPred: " + (", ".join(pred_names) if pred_names else "none"),
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_mismatches(
    model,
    val_dataloader,
    label_cols=label_cols,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    threshold=0.5,
    max_images=12
)